In [2]:
import numpy as np
from imblearn.over_sampling import SMOTE

print("Loading data into notebook memory...")
# Load the raw data from your saved .npy files in the directory
X_train = np.load('X_train.npy')
y_train = np.load('y_train.npy')
X_test = np.load('X_test.npy')
y_test = np.load('y_test.npy')

print("Balancing the training data using SMOTE...")
# Initialize and apply SMOTE
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

print(f"Data loaded and balanced! New training shape: {X_train_smote.shape}")

Loading data into notebook memory...
Balancing the training data using SMOTE...
Data loaded and balanced! New training shape: (454902, 30)


In [3]:
import optuna
import xgboost as xgb
from sklearn.metrics import f1_score
from sklearn.model_selection import StratifiedKFold
import numpy as np
import warnings

# Keep the terminal clean from Optuna's default trial-by-trial spam
optuna.logging.set_verbosity(optuna.logging.WARNING)
warnings.filterwarnings('ignore')

def objective(trial, X, y):
    """
    The objective function that Optuna tries to maximize.
    It builds an XGBoost model with suggested parameters and evaluates it using cross-validation.
    """
    # 1. Define the hyperparameter search space requested for Day 8
    param = {
        'verbosity': 0,
        'objective': 'binary:logistic',
        'eval_metric': 'logloss',
        'random_state': 42,
        # The core parameters to tune:
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        # A reasonable estimator cap to prevent extremely long training times during practice
        'n_estimators': trial.suggest_int('n_estimators', 100, 300)
    }

    # 2. Stratified K-Fold prevents data leakage and maintains your exact class ratios in every fold
    skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    f1_scores = []

    # 3. Cross-Validation Loop
    for train_idx, val_idx in skf.split(X, y):
        
        # FIX: Safely split data whether it is a Pandas DataFrame or a NumPy Array
        if hasattr(X, 'iloc'):
            X_train_fold, X_val_fold = X.iloc[train_idx], X.iloc[val_idx]
            y_train_fold, y_val_fold = y.iloc[train_idx], y.iloc[val_idx]
        else:
            X_train_fold, X_val_fold = X[train_idx], X[val_idx]
            y_train_fold, y_val_fold = y[train_idx], y[val_idx]
        
        model = xgb.XGBClassifier(**param)
        model.fit(X_train_fold, y_train_fold)
        
        # Predict on the validation fold
        preds = model.predict(X_val_fold)
        
        # EXPLICIT OPTIMIZATION: Optimizing strictly for the minority class F1-Score
        score = f1_score(y_val_fold, preds, average='binary', pos_label=1) 
        f1_scores.append(score)

    # Optuna will attempt to MAXIMIZE this return value
    return np.mean(f1_scores)


def run_optimization(X_train, y_train, n_trials=20):
    print(f"Starting Optuna Bayesian Search ({n_trials} trials)...")
    
    # Direction='maximize' because higher F1 is better
    study = optuna.create_study(direction='maximize')
    study.optimize(lambda trial: objective(trial, X_train, y_train), n_trials=n_trials)

    print("\n Optimization Complete.")
    print(f"Best Cross-Validated F1-Score: {study.best_value:.4f}")
    print("Best Parameters Found:")
    for key, value in study.best_params.items():
        if isinstance(value, float):
            print(f"    {key}: {value:.4f}")
        else:
            print(f"    {key}: {value}")
        
    return study.best_params

# ==========================================
# TUNING EXECUTION & MODEL SAVING
# ==========================================

try:
    # 1. Run Optuna (Using your SMOTE training data from previous days)
    best_params = run_optimization(X_train_smote, y_train_smote, n_trials=20)
    
    print("\n Training Final Optimized XGBoost Model...")
    # 2. Train the final model on the ENTIRE training set using the newly found best parameters
    best_xgb_model = xgb.XGBClassifier(**best_params, random_state=42)
    best_xgb_model.fit(X_train_smote, y_train_smote)
    
    print(" Model 'best_xgb_model' is successfully trained and stored in memory!")

except NameError as error_message:
    print("\n CAUGHT ERROR:")
    print(f"[{error_message}]")
    print(" HOW TO FIX THIS:")
    print("Python cannot find your training data variables (e.g. X_train_smote, y_train_smote).")
except Exception as e:
    print(f"\n An unexpected error occurred: {e}")

Starting Optuna Bayesian Search (20 trials)...

 Optimization Complete.
Best Cross-Validated F1-Score: 0.9999
Best Parameters Found:
    max_depth: 7
    learning_rate: 0.2733
    n_estimators: 269

 Training Final Optimized XGBoost Model...
 Model 'best_xgb_model' is successfully trained and stored in memory!


In [4]:
import joblib

joblib.dump(best_xgb_model, 'best_xgb_model.pkl')
print(" Model successfully serialized and exported to disk as 'best_xgb_model.pkl'")

 Model successfully serialized and exported to disk as 'best_xgb_model.pkl'
